# Discover Portugal — Notebook 03: Travel packages 🧳

In notebook 02 we answered 5 questions: **Where, When, Who, What** and **Hidden gems**.
Now we put the answers together: **one package per region**.

| Part of the package | Comes from |
|---|---|
| 🎯 **Theme** of the package | **What**: the specialty of the region (specialty index) |
| 📅 **Best months** and **off-season tip** | **When**: % of the year's stays in each month |
| 🌍 **Countries where to sell it** | **Who**: preference index |
| 📍 **Top places** | **Where**: the places with an official quality marker |
| 💎 **Hidden gems** to add | **Gems** (notebook 02) |
| 🍽️ **You must try** | The traditional dishes of the region (7 Maravilhas da Gastronomia) |

At the end we build a **7-day itinerary** for each package and save the files for the **Streamlit app**.

---
## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CLEAN = "../data/clean/"
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
NOT_COUNTRIES = ["Total", "Portugal", "Foreign countries", "Others"]
YEAR = 2025

pd.set_option("display.max_columns", None)

In [2]:
places = pd.read_csv(CLEAN + "places.csv")
municipalities = pd.read_csv(CLEAN + "tableau_municipalities.csv")   # from notebook 02 (has the hidden gems)
stays_month = pd.read_csv(CLEAN + "stays_month.csv")
stays_nat = pd.read_csv(CLEAN + "stays_nationality.csv")
food = pd.read_csv(CLEAN + "food_products.csv")
dishes = pd.read_csv(CLEAN + "traditional_dishes.csv")

print(len(municipalities), "municipalities |", len(places), "places")

308 municipalities | 7583 places


---
## 1. 🎯 What: the theme of each package

Same **specialty index** as notebook 02 (section 4.1):

> (% of this type of place **in the region**) ÷ (% of this type of place **in Portugal**)

The type with the highest index is the **specialty** of the region, and it gives the **theme** of its package.
(Food is not here: every package has its own typical food, in section 7.)

In [3]:
offer = places.pivot_table(
    index="region", columns="category", values="name", aggfunc="count", fill_value=0)

region_pct = offer.div(offer.sum(axis=1), axis=0)          # % of each type in the region
country_pct = offer.sum() / offer.values.sum()              # % of each type in Portugal
specialty_index = (region_pct / country_pct).round(2)

specialty = pd.DataFrame({"specialty": specialty_index.idxmax(axis=1),
                          "specialty index": specialty_index.max(axis=1)})
specialty.sort_values("specialty index", ascending=False)

,specialty,specialty index
region,,
Açores,nature,5.97
Algarve,beach,4.84
Centro,countryside,4.39
Madeira,viewpoint,3.32
Alentejo,culture,1.25
Lisboa,culture,1.17
Norte,culture,1.10


Each specialty becomes a **theme**:

In [4]:
THEMES = {
    "beach": "🏖️ Beach & Sun",
    "nature": "🌲 Nature Escape",
    "viewpoint": "🔭 Island Views",
    "countryside": "🏡 Historic Villages",
    "culture": "🏛️ Culture & Heritage",
}

specialty["package"] = specialty["specialty"].map(THEMES)
specialty

,specialty,specialty index,package
region,,,
Alentejo,culture,1.25,🏛️ Culture & Heritage
Algarve,beach,4.84,🏖️ Beach & Sun
Açores,nature,5.97,🌲 Nature Escape
Centro,countryside,4.39,🏡 Historic Villages
Lisboa,culture,1.17,🏛️ Culture & Heritage
Madeira,viewpoint,3.32,🔭 Island Views
Norte,culture,1.10,🏛️ Culture & Heritage


💡 Lisboa, Norte and Alentejo all get **Culture & Heritage**, but their index is low (~1.1–1.3):
they are **balanced** regions, with a bit more culture. They are also where most **hidden gems** are (section 5).

---
## 2. 📅 When: best months, off-season tip and when to sell

For each region we rank the months by overnight stays (2025).
- **Peak months** → the 3 busiest months
- **Off-season tip** → the next 3 months: still good, but with fewer tourists
- **When to sell** → **3 months before** the off-season months, so the tourists have time to plan and book
  (3 months is our assumption, not from the data: we don't have data on when tourists book)
- **Best ÷ worst month** → how seasonal the region is (from notebook 02)

In [5]:
region_month = (stays_month[stays_month["year"] == YEAR]
                .pivot_table(index="month_number", columns="region", values="overnight_stays", aggfunc="sum"))
region_month.index = MONTHS

SELL_BEFORE = 3     # months between selling the package and the trip


def in_order(months):
    """Months in the order of the year: ["Oct", "May"] -> "May, Oct" """
    return ", ".join(sorted(months, key=MONTHS.index))


when = {}
for region in region_month.columns:
    ranking = region_month[region].sort_values(ascending=False)
    off_season = list(ranking.index[3:6])
    when[region] = {
        "peak months": in_order(ranking.index[:3]),
        "off-season tip": in_order(off_season),
        "when to sell": in_order({MONTHS[(MONTHS.index(m) - SELL_BEFORE) % 12] for m in off_season}),
        "best ÷ worst month": round(ranking.max() / ranking.min(), 1),
    }

when = pd.DataFrame(when).T
when

,peak months,off-season tip,when to sell,best ÷ worst month
Alentejo,"Jul, Aug, Sep","May, Jun, Oct","Feb, Mar, Jul",4.2
Algarve,"Jul, Aug, Sep","May, Jun, Oct","Feb, Mar, Jul",5.7
Açores,"Jul, Aug, Sep","Apr, May, Jun","Jan, Feb, Mar",4.9
Centro,"Jul, Aug, Sep","May, Jun, Oct","Feb, Mar, Jul",3.1
Lisboa,"Jul, Aug, Oct","May, Jun, Sep","Feb, Mar, Jun",2.0
Madeira,"Jul, Aug, Sep","May, Jun, Oct","Feb, Mar, Jul",1.7
Norte,"Jul, Aug, Sep","May, Jun, Oct","Feb, Mar, Jul",2.8


---
## 3. 🌍 Who: where to sell each package

Same **preference index** as notebook 02 (section 3.3):

> (share of the country **in the region**) ÷ (share of the country **in Portugal**)

Then we keep only countries with at least **500 000 stays** in Portugal (small countries give very big, unstable numbers).
The 3 countries with the highest index are where we sell the package.

In [6]:
foreign = stays_nat[(stays_nat["year"] == YEAR) & (~stays_nat["country"].isin(NOT_COUNTRIES))]
region_country = foreign.pivot_table(index="region", columns="country", values="overnight_stays", aggfunc="sum")

share_in_region = region_country.div(region_country.sum(axis=1), axis=0)
share_in_portugal = region_country.sum() / region_country.values.sum()
preference = (share_in_region / share_in_portugal).round(2)

# Only countries with enough tourists (at least 500 000 stays in Portugal)
big = region_country.sum()[region_country.sum() >= 500_000].index
preference = preference[big]

sell_to = {}
for region in preference.index:
    best = preference.loc[region].sort_values(ascending=False).head(3)
    sell_to[region] = ", ".join(f"{country} ({index:.1f}×)" for country, index in best.items())

pd.Series(sell_to, name="sell to")

Alentejo     Spain (2.0×), Belgium (1.5×), Switzerland (1.5×)
Algarve     Ireland (2.6×), United Kingdom (2.0×), Netherl...
Açores      Canada (2.1×), Switzerland (2.1×), Austria (1.9×)
Centro             Spain (2.6×), Brazil (1.7×), Poland (1.6×)
Lisboa      China (2.0×), Brazil (1.9×), United States (1.7×)
Madeira         Poland (2.9×), Denmark (2.1×), Germany (2.0×)
Norte               Spain (1.9×), Brazil (1.5×), China (1.4×)
Name: sell to, dtype: str

---
## 4. 📍 Where: the top places of each package

For each package we take the places that **match its theme** and have an **official quality marker**.
Most important first: UNESCO → historic villages and parks → National Monuments → the rest.
If two places are equally important, the one in the town with **more places of the theme** comes first.
We show **one place per municipality**, so the package visits different towns.

In [7]:
# Which types of place match each specialty
THEME_CATEGORIES = {
    "beach": ["beach"],
    "nature": ["nature"],
    "viewpoint": ["nature", "viewpoint"],   # Madeira: natural parks + viewpoints
    "countryside": ["countryside"],
    "culture": ["culture"],
}

# Order of importance (lower = more important)
IMPORTANCE = {"UNESCO World Heritage": 0, "Aldeias Históricas de Portugal": 1,
              "National Monument": 2, "Aldeias do Xisto": 2}

top_places = {}
for region, row in specialty.iterrows():
    options = places[(places["region"] == region)
                     & (places["category"].isin(THEME_CATEGORIES[row["specialty"]]))].copy()
    options["importance"] = options["subcategory"].map(IMPORTANCE).fillna(5)
    # Natural areas: the parks come first (name starts with "Parque", or "Natural Park" in English)
    options.loc[options["name"].str.contains("^Parque|Nature Park|Natural Park"), "importance"] = 1
    options["places_in_town"] = options.groupby("municipality")["name"].transform("count")
    options = options.sort_values(["importance", "quality_marker", "places_in_town"], ascending=[True, False, False])
    top = options.drop_duplicates("municipality").head(4)
    top_places[region] = " | ".join(top["name"] + " (" + top["municipality"] + ")")

for region, text in top_places.items():
    print(f"{region}: {text}\n")

Alentejo: Centro Histórico de Évora (Évora) | Cidade Fronteiriça e de Guarnição de Elvas e as suas Fortificações (Elvas) | Parque Mineiro de Aljustrel (Aljustrel) | Fonte das Figueiras (vulgarmente dita «Fonte Mourisca») (Santarém)

Algarve: Salgados (Albufeira) | Zavial (Vila do Bispo) | Vilamoura (Loulé) | Praia da Marina de Portimão (Portimão)

Açores: Parque Natural de São Jorge (Velas) | Nature Park of Santa Maria (Vila do Porto) | Nature Park of Flores (Santa Cruz das Flores) | Nature Park of Faial (Horta)

Centro: Castelo Novo (Fundão) | Piódão (Arganil) | Almeida (Almeida) | Idanha-a-Velha (Idanha-a-Nova)

Lisboa: Mosteiro dos Jerónimos e Torre de Belém (Lisboa) | Paisagem Cultural de Sintra (Sintra) | Palácio Nacional de Mafra e tapada (Mafra) | Portal da Gafaria (Setúbal)

Madeira: Parque Natural da Madeira (Porto Moniz) | Parque Natural Marinho do Cabo Girão (Câmara de Lobos) | Reserva Natural do Sítio da Rocha do Navio (Santana) | Montado do Pereiro (Funchal)

Norte: Centro

---
## 5. 💎 Gems: hidden gems to add to each package

The hidden gems from notebook 02 (many quality markers, few tourists).
Adding them to the packages sends tourists to places that **need** them.

In [8]:
gems = municipalities[municipalities["hidden_gem"] == True].sort_values("n_quality_markers", ascending=False)
gems_by_region = gems.groupby("region")["municipality"].apply(", ".join)
gems_by_region

region
Alentejo                           Moura, Nisa, Borba, Alvito
Açores                                  Santa Cruz das Flores
Centro                            Mangualde, Montemor-o-Velho
Norte       Vila Nova de Foz Côa, Marco de Canaveses, Torr...
Name: municipality, dtype: str

---
## 6. 🧳 The 7 packages

We put everything together.
The **description** is the only part written by hand: a short sentence to sell the package.

In [9]:
DESCRIPTIONS = {
    "Algarve": "Blue Flag beaches, sun and sea in the south of Portugal.",
    "Açores": "Volcanic islands, lakes and nature reserves in the middle of the Atlantic.",
    "Madeira": "Mountains and cliffs over the ocean, with great views all year.",
    "Centro": "Historic and schist villages in the mountains of the Centre of Portugal.",
    "Lisboa": "The capital: monuments, museums and the palaces of Sintra.",
    "Norte": "Porto, Guimarães, Braga, the Minho and the Douro valley.",
    "Alentejo": "Évora, castles and quiet white towns in the plains.",
}

packages = specialty.join(when).join(pd.Series(sell_to, name="sell to"))
packages["top places"] = pd.Series(top_places)
packages["hidden gems"] = gems_by_region
packages["hidden gems"] = packages["hidden gems"].fillna("–")
packages["description"] = pd.Series(DESCRIPTIONS)
packages["overnight stays 2025 (millions)"] = (municipalities.groupby("region")["stays_2025"].sum() / 1_000_000).round(1)

packages = packages.sort_values("specialty index", ascending=False)
packages[["package", "specialty index", "peak months", "off-season tip", "when to sell", "sell to", "hidden gems"]]

,package,specialty index,peak months,off-season tip,when to sell,sell to,hidden gems
region,,,,,,,
Açores,🌲 Nature Escape,5.97,"Jul, Aug, Sep","Apr, May, Jun","Jan, Feb, Mar","Canada (2.1×), Switzerland (2.1×), Austria (1.9×)",Santa Cruz das Flores
Algarve,🏖️ Beach & Sun,4.84,"Jul, Aug, Sep","May, Jun, Oct","Feb, Mar, Jul","Ireland (2.6×), United Kingdom (2.0×), Netherl...",–
Centro,🏡 Historic Villages,4.39,"Jul, Aug, Sep","May, Jun, Oct","Feb, Mar, Jul","Spain (2.6×), Brazil (1.7×), Poland (1.6×)","Mangualde, Montemor-o-Velho"
Madeira,🔭 Island Views,3.32,"Jul, Aug, Sep","May, Jun, Oct","Feb, Mar, Jul","Poland (2.9×), Denmark (2.1×), Germany (2.0×)",–
Alentejo,🏛️ Culture & Heritage,1.25,"Jul, Aug, Sep","May, Jun, Oct","Feb, Mar, Jul","Spain (2.0×), Belgium (1.5×), Switzerland (1.5×)","Moura, Nisa, Borba, Alvito"
Lisboa,🏛️ Culture & Heritage,1.17,"Jul, Aug, Oct","May, Jun, Sep","Feb, Mar, Jun","China (2.0×), Brazil (1.9×), United States (1.7×)",–
Norte,🏛️ Culture & Heritage,1.10,"Jul, Aug, Sep","May, Jun, Oct","Feb, Mar, Jul","Spain (1.9×), Brazil (1.5×), China (1.4×)","Vila Nova de Foz Côa, Marco de Canaveses, Torr..."


---
## 7. 🍽️ You must try: the typical food of each package

Food is part of the trip! Each package gets the **traditional dishes of its region**
(from the *7 Maravilhas da Gastronomia*: the winners first, then the finalists)
and the **local products with a protected name** (DOP / IGP), to buy and take home.

In [10]:
dishes["order"] = dishes["result"].map({"Winner": 0, "Finalist": 1, "Candidate": 2})
dishes["dish"] = dishes["dish"].str.replace(r"\s*\(.*\)", "", regex=True)      # "Alheira de Mirandela (IG)" -> without "(IG)"

must_try_dishes = (dishes.sort_values("order", kind="stable")
                   .groupby("region")["dish"].apply(lambda d: ", ".join(d.head(4))))
# One product of each type (cheese, honey, olive oil, sausages...), the first 3 types
local_products = (food.drop_duplicates(["region", "category"])
                  .groupby("region")["product"].apply(lambda p: ", ".join(p.head(3))))

packages["you must try"] = must_try_dishes
packages["local products"] = local_products
packages[["package", "you must try", "local products"]]

,package,you must try,local products
region,,,
Açores,🌲 Nature Escape,"Polvo assado no forno, Sopas do Espírito Santo...","Ananás dos Açores/São Miguel, Maracujá dos Aço..."
Algarve,🏖️ Beach & Sun,"Xarém com conquilhas, Muxama de atum, Percebes...","Mel da Serra de Monchique, Sal de Tavira ou Fl..."
Centro,🏡 Historic Villages,"Queijo Serra da Estrela, Leitão da Bairrada, C...",Azeites da Beira Interior / Azeite da Beira A...
Madeira,🔭 Island Views,"Coelho do Porto Santo à Caçador, Bolo do Caco,...",Anona da Madeira
Alentejo,🏛️ Culture & Heritage,"Açorda à Alentejana, Espargos com ovos, Pezinh...","Azeite de Moura, Azeitonas de Conserva de Elva..."
Lisboa,🏛️ Culture & Heritage,"Sardinha assada, Arroz de marisco, Pastel de B...","Maçã Riscadinha de Palmela, Queijo de Azeitão"
Norte,🏛️ Culture & Heritage,"Alheira de Mirandela, Caldo verde, Bacalhau à ...","Azeite de Trás-os-Montes, Azeitona de conserva..."


💡 In the itineraries (section 8), each day also gets **something to try, never repeated in the same trip**: first the dishes
with the name of that town (e.g. *Ovos moles de Aveiro* in Aveiro), then the other dishes of the region, and then the local products.

🔭 **Next step: a Food Route.** Portugal is famous for its food and its high-quality products, so food could be a package of its own.
We tried a route from North to South, but it was too complex for this project (incomplete restaurant data, and a good route
also needs markets, wineries and festivals). For now, every package tells you what you **must try**.

---
## 8. 🗓️ A 7-day itinerary for each package

A package is more useful with a **day-by-day plan**. We build it with simple rules
(the code is in `app/itinerary.py`, so the Streamlit app uses exactly the same rules for any number of days):

| # | Rule |
|---|---|
| 1 | **Islands:** the trip stays on **one island** (the one with the most places of the theme) |
| 2 | **Beach & Sun:** no jumping around: we stay in **one town** (the one with the most Blue Flag beaches) and go to a different beach every day (less than 40 km away), the closest first |
| 3 | **Other packages:** a road trip. We start in the town with the most places of the theme (**UNESCO sites and historic villages count 3 times**), and each day we go to the **best town less than 60 km away** |
| 4 | **Hidden gems:** in trips of 5 days or more, one day goes to the hidden gem closest to the start (if it's less than 150 km away) |
| 5 | **Places per day** depend on the theme: **1 beach** (a beach takes the whole day), **2** natural areas, **2** historic villages, **3** viewpoints, **4** monuments and museums. If the town has fewer, we add places of the theme in the **towns around it** (less than 25 km away). Only places that **interest tourists**: no small churches, chapels, pillories, fountains, houses… (list in `itinerary.py`) |
| 6 | More days than towns (small islands): an extra day in the town with the most places still to see |
| 7 | **Order:** we put the towns in the order with the **shortest round trip** (back to the start at the end, e.g. the airport), so we never go back and forth |
| 8 | In the app, the tourist can **choose where to start** (e.g. Óbidos) and **skip towns**: the ones already visited, and the busiest half of the region if they prefer fewer tourists |
| 🍽️ | Each day: **dishes to try, never repeated in the same trip** (the dishes of that town first, then the region's, then the local DOP/IGP products) |

In [11]:
import sys
sys.path.append("../app")          # the itinerary code is in the app folder
from itinerary import build_itinerary, trip_food

DAYS = 7
raw_dishes = pd.read_csv(CLEAN + "traditional_dishes.csv")    # the original names (must_try cleans them itself)
itineraries = []
for _, package in packages.reset_index().iterrows():     # reset_index: the region becomes a column
    plan = build_itinerary(places, municipalities, package, DAYS)
    plan.insert(0, "package", f"{package['package']} · {package['region']}")
    plan["you must try"] = trip_food(raw_dishes, food, package["region"], list(plan["town"]))
    itineraries.append(plan)

itineraries = pd.concat(itineraries, ignore_index=True)[["package", "day", "town", "places", "you must try", "note", "island"]]
print(itineraries["package"].value_counts().sort_index())

package
🌲 Nature Escape · Açores            7
🏖️ Beach & Sun · Algarve            7
🏛️ Culture & Heritage · Alentejo    7
🏛️ Culture & Heritage · Lisboa      7
🏛️ Culture & Heritage · Norte       7
🏡 Historic Villages · Centro        7
🔭 Island Views · Madeira            7
Name: count, dtype: int64


---
## 9. 📋 The package sheets

Each package as we would **send it to a travel agency**: the key facts and the day-by-day plan.
Everything is in **English**, like in the app (with `app/english.py`): the places, the food and the regions
(*Norte* → *North*, *Açores* → *Azores*...). Only the **town names stay in Portuguese**.
(In the data we keep the official INE names of the regions.)

In [12]:
from IPython.display import Markdown, display
from english import english_foods, english_name


def package_sheet(region):
    """The package sheet for a travel agency: the key facts and the 7-day plan"""
    p = packages.loc[region]
    plan = itineraries[itineraries["package"] == f"{p['package']} · {region}"]
    category = "beach" if p["specialty"] == "beach" else None
    island = plan["island"].dropna()

    # 1. The key facts
    text = f"""
*{p['description']}*

| | |
|---|---|
| 📅 **Best time to travel** | {p['off-season tip']} (fewer tourists than in the peak: {p['peak months']}) |
| 📣 **When to sell** | {p['when to sell']} (3 months before the trip) |
| 🌍 **Sell in** | {p['sell to']} |
| 💎 **Hidden gems** | {p['hidden gems']} |
| 🍽️ **You must try** | {english_foods(p['you must try'])} |
| 🧀 **Take home** | {english_foods(p['local products'])} |
| 🏨 **Overnight stays in 2025** | {p['overnight stays 2025 (millions)']} million |

**🗓️ {DAYS} days{' · island of ' + island.iloc[0] if len(island) else ''}**

| Day | Town | What to see | You must try | Note |
|---|---|---|---|---|
"""
    # 2. The day-by-day plan
    for _, day in plan.iterrows():
        see = " · ".join(english_name(place, category) for place in day["places"].split(" · "))
        note = day["note"] if isinstance(day["note"], str) else ""
        text += f"| {day['day']} | **{day['town']}** | {see} | {english_foods(day['you must try'])} | {note} |\n"
    display(Markdown(text))

---
### 9.1 🌲 Nature Escape · Azores

In [13]:
package_sheet("Açores")


*Volcanic islands, lakes and nature reserves in the middle of the Atlantic.*

| | |
|---|---|
| 📅 **Best time to travel** | Apr, May, Jun (fewer tourists than in the peak: Jul, Aug, Sep) |
| 📣 **When to sell** | Jan, Feb, Mar (3 months before the trip) |
| 🌍 **Sell in** | Canada (2.1×), Switzerland (2.1×), Austria (1.9×) |
| 💎 **Hidden gems** | Santa Cruz das Flores |
| 🍽️ **You must try** | Roasted octopus, Holy Spirit soup, Boiled barnacles, Grilled Azores limpets |
| 🧀 **Take home** | Azores pineapple, Azores passion fruit, Azores honey |
| 🏨 **Overnight stays in 2025** | 3.1 million |

**🗓️ 7 days · island of São Miguel**

| Day | Town | What to see | You must try | Note |
|---|---|---|---|---|
| 1 | **Ponta Delgada** | Natural Monument of Pico das Camarinhas e Ponta da Ferraria · Protected Landscape of Sete Cidades | Roasted octopus |  |
| 2 | **Ponta Delgada** | Cave of Carvão · Forest Reserve of Recreio do Pinhal da Paz | Holy Spirit soup | 🔁 More to see |
| 3 | **Ponta Delgada** | Ponta da Bretanha · Feteiras | Boiled barnacles | 🔁 More to see |
| 4 | **Ribeira Grande** | Natural Monument of Caldeira Velha · Ponta do Cintrão-Ponta da Maia | Grilled Azores limpets |  |
| 5 | **Povoação** | Forest Reserve of Pico da Vara · Ponta do Arnel | Terceira beef pot roast (alcatra) |  |
| 6 | **Vila Franca do Campo** | Caloura-Ilhéu de Vila Franca do Campo · Serra de Água de Pau | Furnas volcanic stew |  |
| 7 | **Lagoa (Açores)** | Lagoa do Fogo | Azores pineapple |  |


---
### 9.2 🏖️ Beach & Sun · Algarve

In [14]:
package_sheet("Algarve")


*Blue Flag beaches, sun and sea in the south of Portugal.*

| | |
|---|---|
| 📅 **Best time to travel** | May, Jun, Oct (fewer tourists than in the peak: Jul, Aug, Sep) |
| 📣 **When to sell** | Feb, Mar, Jul (3 months before the trip) |
| 🌍 **Sell in** | Ireland (2.6×), United Kingdom (2.0×), Netherlands (1.5×) |
| 💎 **Hidden gems** | – |
| 🍽️ **You must try** | Corn porridge with clams (xarém), Dried tuna (muxama), Aljezur goose barnacles, Algarve-style partridge |
| 🧀 **Take home** | Monchique honey, Tavira salt, Aljezur sweet potatoes |
| 🏨 **Overnight stays in 2025** | 20.8 million |

**🗓️ 7 days**

| Day | Town | What to see | You must try | Note |
|---|---|---|---|---|
| 1 | **Albufeira** | Alemães Beach | Corn porridge with clams (xarém) | 🏖️ Beach day · stay in Albufeira |
| 2 | **Albufeira** | INATEL – Albufeira Beach | Dried tuna (muxama) | 🏖️ Beach day · stay in Albufeira |
| 3 | **Albufeira** | Rocha Baixinha Beach | Aljezur goose barnacles | 🏖️ Beach day · stay in Albufeira |
| 4 | **Albufeira** | Peneco Beach | Algarve-style partridge | 🏖️ Beach day · stay in Albufeira |
| 5 | **Albufeira** | Aveiros Beach | Dom Rodrigo (egg and almond sweet) | 🏖️ Beach day · stay in Albufeira |
| 6 | **Albufeira** | Oura Beach | Monchique honey | 🏖️ Beach day · stay in Albufeira |
| 7 | **Albufeira** | Oura-Leste Beach | Tavira salt | 🏖️ Beach day · stay in Albufeira |


---
### 9.3 🏡 Historic Villages · Centre

In [15]:
package_sheet("Centro")


*Historic and schist villages in the mountains of the Centre of Portugal.*

| | |
|---|---|
| 📅 **Best time to travel** | May, Jun, Oct (fewer tourists than in the peak: Jul, Aug, Sep) |
| 📣 **When to sell** | Feb, Mar, Jul (3 months before the trip) |
| 🌍 **Sell in** | Spain (2.6×), Brazil (1.7×), Poland (1.6×) |
| 💎 **Hidden gems** | Mangualde, Montemor-o-Velho |
| 🍽️ **You must try** | Serra da Estrela cheese, Bairrada roast suckling pig, Chanfana (goat stewed in red wine), Hunter's rabbit stew |
| 🧀 **Take home** | Beira Interior olive oil, Marinhoa beef, Serra da Estrela lamb |
| 🏨 **Overnight stays in 2025** | 8.5 million |

**🗓️ 7 days**

| Day | Town | What to see | You must try | Note |
|---|---|---|---|---|
| 1 | **Almeida** | Almeida · Castelo Mendo | Serra da Estrela cheese, Bairrada roast suckling pig |  |
| 2 | **Mangualde** | Dolmen of Cunha Baixa · Iron Age hillfort of Bom Sucesso | Chanfana (goat stewed in red wine), Hunter's rabbit stew | 💎 Hidden gem |
| 3 | **Arganil** | Piódão · Benfeita | Partridge in escabeche, Tentúgal pastries |  |
| 4 | **Góis** | Aigra Nova · Aigra Velha | Figueira fish soup, Figueira shrimps |  |
| 5 | **Fundão** | Castelo Novo · Barroca | Aveiro soft eggs (ovos moles), Beira Interior olive oil |  |
| 6 | **Idanha-a-Nova** | Idanha-a-Velha · Monsanto | Marinhoa beef, Serra da Estrela lamb |  |
| 7 | **Belmonte** | Belmonte | Esmolfe apples, Rocha pears from the Oeste |  |


---
### 9.4 🔭 Island Views · Madeira

In [16]:
package_sheet("Madeira")


*Mountains and cliffs over the ocean, with great views all year.*

| | |
|---|---|
| 📅 **Best time to travel** | May, Jun, Oct (fewer tourists than in the peak: Jul, Aug, Sep) |
| 📣 **When to sell** | Feb, Mar, Jul (3 months before the trip) |
| 🌍 **Sell in** | Poland (2.9×), Denmark (2.1×), Germany (2.0×) |
| 💎 **Hidden gems** | – |
| 🍽️ **You must try** | Porto Santo hunter's rabbit, Bolo do caco (Madeira flatbread), Madeira limpets, Madeira tuna steak |
| 🧀 **Take home** | Madeira custard apple |
| 🏨 **Overnight stays in 2025** | 9.9 million |

**🗓️ 7 days · island of Madeira**

| Day | Town | What to see | You must try | Note |
|---|---|---|---|---|
| 1 | **Santana** | Nature Reserve of Sítio da Rocha do Navio · Arco de São Jorge · Queimadas | Porto Santo hunter's rabbit |  |
| 2 | **São Vicente** | Pico Casado · Chão dos Louros · Bica da Cana | Bolo do caco (Madeira flatbread) |  |
| 3 | **Porto Moniz** | Natural Park of Madeira · Santinha · Aduelas | Madeira limpets |  |
| 4 | **Calheta (Madeira)** | Rabaçal · Paul · Risco | Madeira tuna steak |  |
| 5 | **Câmara de Lobos** | Natural Monument of Cabo Girão · Teleférico do Rancho · Protected Landscape of Cabo Girão | Beef skewers on bay laurel sticks |  |
| 6 | **Funchal** | Montado do Pereiro · Pico dos Barcelos · Montado do Barreiro | Madeira honey cake |  |
| 7 | **Machico** | Ponta de São Lourenço · Ribeiro Frio · Ponta do Rosto | Madeira custard apple |  |


---
### 9.5 🏛️ Culture & Heritage · Lisbon

In [17]:
package_sheet("Lisboa")


*The capital: monuments, museums and the palaces of Sintra.*

| | |
|---|---|
| 📅 **Best time to travel** | May, Jun, Sep (fewer tourists than in the peak: Jul, Aug, Oct) |
| 📣 **When to sell** | Feb, Mar, Jun (3 months before the trip) |
| 🌍 **Sell in** | China (2.0×), Brazil (1.9×), United States (1.7×) |
| 💎 **Hidden gems** | – |
| 🍽️ **You must try** | Grilled sardines, Seafood rice, Pastel de Belém (custard tart), Codfish cakes |
| 🧀 **Take home** | Riscadinha de Palmela apples, Azeitão cheese |
| 🏨 **Overnight stays in 2025** | 21.2 million |

**🗓️ 7 days**

| Day | Town | What to see | You must try | Note |
|---|---|---|---|---|
| 1 | **Lisboa** | Jerónimos Monastery and Belém Tower · Parque do Monteiro-Mor · Cordoaria Nacional · Faculdade de Direito da Universidade de Lisboa | Grilled sardines |  |
| 2 | **Lisboa** | Church of Memória · Church of Sagrado Coração de Jesus · Jardim da Fundação Calouste Gulbenkian · Palácio Vale Flor | Seafood rice | 🔁 More to see |
| 3 | **Lisboa** | Palace of Almadas · Palácio Calheta · Reservatório da Mãe d'Água das Amoreiras · Picadeiro Real de Belém | Pastel de Belém (custard tart) | 🔁 More to see |
| 4 | **Mafra** | National Palace of Mafra · Church of Santo André · Fort of Milreu · Ponte antiga em Cheleiros | Codfish cakes |  |
| 5 | **Sintra** | Cultural Landscape of Sintra · Parque de Monserrate · Castle of Mouros · National Palace of Queluz | Stone soup |  |
| 6 | **Cascais** | Forte Novo · Fort of Crismina · Fort of Nossa Senhora da Conceição · Fort of Nossa Senhora da Guia | Clams with garlic and coriander |  |
| 7 | **Setúbal** | Palace of Bacalhoa · Palácio e Quinta da Bacalhoa · Church of antigo Mosteiro de Jesus e claustro · Church of São Julião de Setúbal | Bacalhau à Brás (cod with eggs and potatoes) |  |


---
### 9.6 🏛️ Culture & Heritage · North

In [18]:
package_sheet("Norte")


*Porto, Guimarães, Braga, the Minho and the Douro valley.*

| | |
|---|---|
| 📅 **Best time to travel** | May, Jun, Oct (fewer tourists than in the peak: Jul, Aug, Sep) |
| 📣 **When to sell** | Feb, Mar, Jul (3 months before the trip) |
| 🌍 **Sell in** | Spain (1.9×), Brazil (1.5×), China (1.4×) |
| 💎 **Hidden gems** | Vila Nova de Foz Côa, Marco de Canaveses, Torre de Moncorvo |
| 🍽️ **You must try** | Mirandela alheira (smoked sausage), Caldo verde (kale soup), Bacalhau à Gomes de Sá (baked cod), Porto-style tripe |
| 🧀 **Take home** | Trás-os-Montes olive oil, Negrinha de Freixo olives, Arouquesa beef |
| 🏨 **Overnight stays in 2025** | 14.7 million |

**🗓️ 7 days**

| Day | Town | What to see | You must try | Note |
|---|---|---|---|---|
| 1 | **Porto** | Historic Centre of Porto · Cadeia da Relação · Parque de Serralves · Church of Carmo | Porto-style tripe, Mirandela alheira (smoked sausage) |  |
| 2 | **Marco de Canaveses** | Church of Santo André · Church of Vila Boa do Bispo · Church of Santo Isidoro · Tongóbriga | Caldo verde (kale soup), Bacalhau à Gomes de Sá (baked cod) | 💎 Hidden gem |
| 3 | **Guimarães** | Historic Centre of Guimarães · Zona de Couros · Citânia de Sabroso · Citânia de Briteiros | Abade de Priscos pudding, Lamego meat bread |  |
| 4 | **Arcos de Valdevez** | Antas da Serra de Soajo · Paço da Giela · Sanctuary of Nossa Senhora da Peneda · Paisagem Cultural de Sistelo | Chestnut soup, Lamprey rice |  |
| 5 | **Ponte de Lima** | Bridge of Ponte de Lima · Church of Misericórdia de Ponte de Lima · Monastery of Refoios do Lima · Torres de São Paulo e da Cadeia e o pequeno pano de muralha existente entre as duas torres | Bacalhau à Zé do Pipo (cod with mashed potatoes), Mirandela alheira with turnip greens |  |
| 6 | **Viana do Castelo** | Paços Municipais de Viana do Castelo · Misericórdia de Viana do Castelo · Church of São Cláudio · Church of Santa Cruz | Mirandesa beef steak, Trás-os-Montes olive oil |  |
| 7 | **Braga** | Sanctuary of Bom Jesus do Monte · Sete Fontes de S. Victor · Monastery of São Martinho de Tibães · Arco da Porta Nova | Bragançano lamb, Negrinha de Freixo olives |  |


---
### 9.7 🏛️ Culture & Heritage · Alentejo

In [19]:
package_sheet("Alentejo")


*Évora, castles and quiet white towns in the plains.*

| | |
|---|---|
| 📅 **Best time to travel** | May, Jun, Oct (fewer tourists than in the peak: Jul, Aug, Sep) |
| 📣 **When to sell** | Feb, Mar, Jul (3 months before the trip) |
| 🌍 **Sell in** | Spain (2.0×), Belgium (1.5×), Switzerland (1.5×) |
| 💎 **Hidden gems** | Moura, Nisa, Borba, Alvito |
| 🍽️ **You must try** | Alentejo bread soup, Wild asparagus with eggs, Pig's trotters with coriander, Barrancos cured ham |
| 🧀 **Take home** | Moura olive oil, Elvas olives, Alentejo beef |
| 🏨 **Overnight stays in 2025** | 3.7 million |

**🗓️ 7 days**

| Day | Town | What to see | You must try | Note |
|---|---|---|---|---|
| 1 | **Évora** | Historic Centre of Évora · Walls of Évora · Porta de Aviz · Convent of Lóios | Évora cheese, Alentejo bread soup |  |
| 2 | **Alvito** | Manor house of Água de Peixes · Church of Nossa Senhora da Assunção · Castle of Alvito · Church of Misericórdia de Alvito e Capela de Nossa Senhora das Candeias | Wild asparagus with eggs, Pig's trotters with coriander | 💎 Hidden gem |
| 3 | **Montemor-o-Novo** | Church of São João Baptista · Anta Grande da Comenda da Igreja · Cave of Escoural · Convent of Nossa Senhora da Saudação | Montemor-o-Novo lamb, Barrancos cured ham |  |
| 4 | **Castelo de Vide** | Menir da Meada · Castle of Castelo de Vide · Dolmen of Casa dos Galhardos · Dolmen of Tapadas de Pedro Álvaro | Gazpacho with fried mackerel, Dogfish soup |  |
| 5 | **Portalegre** | Church of São Bernardo · Castle of Alegrete · Dois claustros existentes no convento anexo à Igreja de São Bernardo · Castle of Portalegre | São Julião-Portalegre cherries, Marvão-Portalegre chestnuts |  |
| 6 | **Elvas** | Garrison Border Town of Elvas and its Fortifications · Fort of Nossa Senhora da Graça · Fort of Santa Luzia · Church of São Domingos | Elvas olives, Elvas plums |  |
| 7 | **Estremoz** | Villa lusitano-romana de Santa Vitória do Ameixial · Antiga Casa da Câmara · Claustro da Misericórdia de Estremoz · Castle of Estremoz | Estremoz chouriço sausage, Estremoz chouriço sausage |  |


---
## 10. 💾 Save the files for the Streamlit app

In [20]:
# The 7 packages and their 7-day itineraries
packages.index.name = "region"
packages.reset_index().to_csv(CLEAN + "packages.csv", index=False)
itineraries.to_csv(CLEAN + "itineraries_7_days.csv", index=False)

# % of the year in each month, by municipality (the app says if a month is busy or quiet)
mun_month = (stays_month[stays_month["year"] == YEAR]
             .pivot_table(index="municipality", columns="month_number", values="overnight_stays", aggfunc="sum"))
mun_month = (mun_month.div(mun_month.sum(axis=1), axis=0) * 100).round(2)
mun_month.columns = MONTHS
mun_month.reset_index().to_csv(CLEAN + "municipality_month_pct.csv", index=False)

print("Saved ✅")

Saved ✅


---
## ✅ Summary

- **7 packages, one per region**, built from the answers of notebook 02:
  theme (**What**), best months (**When**), countries where to sell (**Who**), top places (**Where**) and **hidden gems**
- **🍽️ You must try:** the typical dishes and local products of each region
- A **7-day itinerary** for each package (simple rules: one island, one base town for the beach, road trips < 60 km, a day in a hidden gem, shortest order)
- **Places per day by theme**: 1 beach, but up to 4 monuments in a historic town (completed with the towns around)
- A **package sheet** for each region, ready to send to a travel agency
- Every number comes from the data; only the short descriptions are written by hand
- The files are ready for the **Streamlit app** (`app/app.py`)